In [2]:
# ========================================
# ERROR ANALYSIS - SMEAL Project
# Confusion Matrix, False Positives/Negatives, Class Performance
# ========================================

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from pathlib import Path
import json
import cv2
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ========================================
# 1. CHARGER LE MODÈLE PRÉ-ENTRAÎNÉ
# ========================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Charger les noms des classes Food-101
food101_classes = [
    'apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare',
    'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito',
    'bruschetta', 'caesar_salad', 'cannoli', 'caprese_salad', 'carrot_cake',
    'ceviche', 'cheese_plate', 'cheesecake', 'chicken_curry', 'chicken_parmesan',
    'chicken_quesadilla', 'chicken_wings', 'chocolate_cake', 'chocolate_mousse', 'churros',
    'clam_chowder', 'club_sandwich', 'crab_cakes', 'creme_brulee', 'croque_madame',
    'cup_cakes', 'cured_ham', 'curry_soup', 'custard', 'cuts_of_meat',
    'deviled_eggs', 'dim_sum', 'donuts', 'dumplings', 'edamame',
    'eggs_benedict', 'escargots', 'falafel', 'fall_salad', 'fettuccine_alfredo',
    'filet_mignon', 'fish_and_chips', 'fish_tacos', 'flashy_fish', 'foie_gras',
    'french_fries', 'french_onion_soup', 'french_toast', 'fried_calamari', 'fried_rice',
    'fried_shrimp', 'frozen_yogurt', 'fruit_salad', 'garlic_bread', 'gnocchi',
    'greek_salad', 'green_beans', 'green_curry', 'grilled_cheese_sandwich', 'grilled_salmon',
    'grits', 'guacamole', 'gyoza', 'halibut', 'ham_and_cheese_sandwich',
    'hamburger', 'hand_pulled_pork', 'handmade_pasta', 'hankook_guksu', 'hash_browns',
    'homemade_guacamole', 'hot_and_sour_soup', 'hot_dog', 'huevos_rancheros', 'hummus',
    'ice_cream', 'idli', 'indian_curry', 'irish_coffee', 'italian_wedding_soup',
    'japanese_curry', 'jerk_chicken', 'joumou_soup', 'juice', 'julep',
]

print(f"✅ {len(food101_classes)} classes Food-101 chargées")

# ========================================
# 2. CHARGER LE MODÈLE ResNet-18
# ========================================

def load_model(model_path, num_classes=101):
    """Charger le modèle ResNet-18 fine-tuné"""
    model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=False)
    model.fc = nn.Linear(512, num_classes)
    
    # Charger les poids
    if Path(model_path).exists():
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint)
        print(f"✅ Modèle chargé de: {model_path}")
    else:
        print(f"⚠️  Fichier modèle non trouvé: {model_path}")
        print("   → Utilisation du modèle non entraîné (pour démo)")
    
    model = model.to(device)
    model.eval()
    return model

# Remplacer par le chemin de votre modèle
model_path = "C:/SmartMeal/models/resnet18_food101.pth"
model = load_model(model_path)

# ========================================
# 3. CRÉER UN DATASET DE DÉMONSTRATION
# ========================================

class FoodDatasetDummy(Dataset):
    """Dataset de démonstration avec images synthétiques"""
    
    def __init__(self, num_samples=100, num_classes=101):
        self.num_samples = num_samples
        self.num_classes = num_classes
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])
        ])
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        # Créer une image synthétique
        img = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
        label = idx % self.num_classes
        img = Image.fromarray(img)
        img = self.transform(img)
        return img, label

# Créer le dataset
test_dataset = FoodDatasetDummy(num_samples=200, num_classes=101)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"✅ Dataset de test créé: {len(test_dataset)} images")

# ========================================
# 4. GÉNÉRER LES PRÉDICTIONS
# ========================================

def generate_predictions(model, test_loader, device):
    """Générer toutes les prédictions"""
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    all_images = []
    
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_images.extend(images.cpu().numpy())
            
            if (batch_idx + 1) % 3 == 0:
                print(f"  ✓ Batch {batch_idx + 1}/{len(test_loader)}")
    
    return (np.array(all_preds), np.array(all_labels), 
            np.array(all_probs), np.array(all_images))

print("\n🔄 Génération des prédictions...")
all_preds, all_labels, all_probs, all_images = generate_predictions(model, test_loader, device)
print(f"✅ {len(all_preds)} prédictions générées")

# ========================================
# 5. CONFUSION MATRIX
# ========================================

def plot_confusion_matrix(y_true, y_pred, class_names, top_n=15, save_path=None):
    """Visualiser la confusion matrix pour les top N classes"""
    
    print("\n" + "="*60)
    print("CONFUSION MATRIX")
    print("="*60)
    
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    
    # Sélectionner les top N classes (les plus fréquentes)
    class_freqs = np.bincount(y_true, minlength=len(class_names))
    top_indices = np.argsort(class_freqs)[-top_n:]
    
    cm_top = cm[np.ix_(top_indices, top_indices)]
    top_classes = [class_names[i] for i in top_indices]
    
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm_top, annot=True, fmt='d', cmap='Blues',
                xticklabels=top_classes, yticklabels=top_classes,
                cbar_kws={'label': 'Count'}, square=True)
    plt.title(f'Confusion Matrix - Top {top_n} Classes\nResNet-18 Food-101', 
              fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✅ Sauvegardé: {save_path}")
    
    plt.show()
    return cm

cm = plot_confusion_matrix(all_labels, all_preds, food101_classes, 
                           top_n=15, save_path="confusion_matrix.png")

# ========================================
# 6. CLASSIFICATION REPORT
# ========================================

def print_classification_report(y_true, y_pred, class_names, top_k=20):
    """Rapport détaillé par classe"""
    
    print("\n" + "="*80)
    print("CLASSIFICATION REPORT - TOP CLASSES")
    print("="*80)
    
    report = classification_report(y_true, y_pred, 
                                   target_names=class_names,
                                   digits=4, zero_division=0)
    print(report)
    
    # Sauvegarder
    with open("classification_report.txt", "w") as f:
        f.write(report)
    print(f"✅ Rapport sauvegardé: classification_report.txt")
    
    return report

report = print_classification_report(all_labels, all_preds, food101_classes)

# ========================================
# 7. VISUALISER LES ERREURS
# ========================================

def visualize_misclassifications(images, y_true, y_pred, probs, class_names, 
                                num_samples=12, save_path=None):
    """Afficher les False Positives et False Negatives"""
    
    print("\n" + "="*60)
    print("VISUALIZING MISCLASSIFICATIONS")
    print("="*60)
    
    mistakes = y_true != y_pred
    mistake_indices = np.where(mistakes)[0]
    
    if len(mistake_indices) == 0:
        print("✅ Aucune erreur trouvée !")
        return
    
    print(f"Total d'erreurs: {len(mistake_indices)} / {len(y_true)} ({len(mistake_indices)/len(y_true)*100:.2f}%)")
    
    num_samples = min(num_samples, len(mistake_indices))
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.ravel()
    
    for idx, ax_idx in enumerate(mistake_indices[:num_samples]):
        img = images[ax_idx]
        true_label = class_names[y_true[ax_idx]]
        pred_label = class_names[y_pred[ax_idx]]
        confidence = probs[ax_idx, y_pred[ax_idx]]
        
        # Dénormaliser l'image
        if img.shape[0] == 3:  # Format (C, H, W)
            img = img.transpose(1, 2, 0)
        
        # Dénormaliser (inverse du normalization ImageNet)
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)
        
        axes[idx].imshow(img)
        axes[idx].set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.2%})',
                           color='red', fontweight='bold', fontsize=9)
        axes[idx].axis('off')
    
    # Cacher les axes non utilisés
    for idx in range(num_samples, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('Misclassifications - ResNet-18', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✅ Sauvegardé: {save_path}")
    
    plt.show()

visualize_misclassifications(all_images, all_labels, all_preds, all_probs, 
                            food101_classes, num_samples=12,
                            save_path="misclassifications.png")

# ========================================
# 8. ANALYSE PAR CLASSE - WORST CLASSES
# ========================================

def worst_performing_classes(y_true, y_pred, class_names, top_k=15):
    """Les classes les plus mal reconnues"""
    
    print("\n" + "="*70)
    print("WORST PERFORMING CLASSES")
    print("="*70)
    
    class_accuracies = {}
    class_counts = {}
    
    for class_idx in range(len(class_names)):
        mask = y_true == class_idx
        if mask.sum() == 0:
            continue
        
        accuracy = (y_pred[mask] == class_idx).mean()
        count = mask.sum()
        class_accuracies[class_names[class_idx]] = accuracy
        class_counts[class_names[class_idx]] = count
    
    # Trier par accuracy
    sorted_classes = sorted(class_accuracies.items(), key=lambda x: x[1])
    
    print(f"\n{'Rank':<5} {'Class':<30} {'Accuracy':<12} {'Samples':<10}")
    print("-" * 57)
    for rank, (cls, acc) in enumerate(sorted_classes[:top_k], 1):
        count = class_counts[cls]
        print(f"{rank:<5} {cls:<30} {acc:<12.2%} {count:<10}")
    
    # Visualiser
    classes_worst, accs_worst = zip(*sorted_classes[:top_k])
    
    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.barh(classes_worst, accs_worst, color='coral', edgecolor='darkred', linewidth=1.5)
    ax.set_xlabel('Accuracy', fontsize=12, fontweight='bold')
    ax.set_title(f'Bottom {top_k} Classes - Lowest Accuracy\nResNet-18 Food-101',
                fontsize=14, fontweight='bold')
    ax.set_xlim([0, 1])
    
    # Ajouter les valeurs sur les barres
    for bar, acc in zip(bars, accs_worst):
        ax.text(acc + 0.02, bar.get_y() + bar.get_height()/2,
               f'{acc:.1%}', va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('worst_classes.png', dpi=300, bbox_inches='tight')
    print(f"\n✅ Graphique sauvegardé: worst_classes.png")
    plt.show()
    
    return sorted_classes

worst_classes = worst_performing_classes(all_labels, all_preds, food101_classes, top_k=15)

# ========================================
# 9. BEST CLASSES
# ========================================

def best_performing_classes(y_true, y_pred, class_names, top_k=15):
    """Les classes les mieux reconnues"""
    
    print("\n" + "="*70)
    print("BEST PERFORMING CLASSES")
    print("="*70)
    
    class_accuracies = {}
    class_counts = {}
    
    for class_idx in range(len(class_names)):
        mask = y_true == class_idx
        if mask.sum() == 0:
            continue
        
        accuracy = (y_pred[mask] == class_idx).mean()
        count = mask.sum()
        class_accuracies[class_names[class_idx]] = accuracy
        class_counts[class_names[class_idx]] = count
    
    # Trier par accuracy (décroissant)
    sorted_classes = sorted(class_accuracies.items(), key=lambda x: x[1], reverse=True)
    
    print(f"\n{'Rank':<5} {'Class':<30} {'Accuracy':<12} {'Samples':<10}")
    print("-" * 57)
    for rank, (cls, acc) in enumerate(sorted_classes[:top_k], 1):
        count = class_counts[cls]
        print(f"{rank:<5} {cls:<30} {acc:<12.2%} {count:<10}")
    
    # Visualiser
    classes_best, accs_best = zip(*sorted_classes[:top_k])
    
    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.barh(classes_best, accs_best, color='lightgreen', edgecolor='darkgreen', linewidth=1.5)
    ax.set_xlabel('Accuracy', fontsize=12, fontweight='bold')
    ax.set_title(f'Top {top_k} Classes - Highest Accuracy\nResNet-18 Food-101',
                fontsize=14, fontweight='bold')
    ax.set_xlim([0, 1])
    
    # Ajouter les valeurs
    for bar, acc in zip(bars, accs_best):
        ax.text(acc - 0.05, bar.get_y() + bar.get_height()/2,
               f'{acc:.1%}', va='center', ha='right', fontsize=10, fontweight='bold', color='white')
    
    plt.tight_layout()
    plt.savefig('best_classes.png', dpi=300, bbox_inches='tight')
    print(f"\n✅ Graphique sauvegardé: best_classes.png")
    plt.show()
    
    return sorted_classes

best_classes = best_performing_classes(all_labels, all_preds, food101_classes, top_k=15)

# ========================================
# 10. RÉSUMÉ GLOBAL
# ========================================

def print_summary(y_true, y_pred):
    """Résumé des performances globales"""
    
    accuracy = accuracy_score(y_true, y_pred)
    total_samples = len(y_true)
    total_errors = (y_true != y_pred).sum()
    
    print("\n" + "="*70)
    print("GLOBAL SUMMARY")
    print("="*70)
    print(f"Total Samples:        {total_samples}")
    print(f"Correct Predictions:  {total_samples - total_errors} ({(1-total_errors/total_samples)*100:.2f}%)")
    print(f"Incorrect Predictions: {total_errors} ({total_errors/total_samples*100:.2f}%)")
    print(f"Overall Accuracy:     {accuracy:.4f} ({accuracy*100:.2f}%)")
    print("="*70)

print_summary(all_labels, all_preds)

print("\n✅ ERROR ANALYSIS COMPLÈTE !")
print("\n📊 Fichiers générés:")
print("  • confusion_matrix.png")
print("  • misclassifications.png")
print("  • worst_classes.png")
print("  • best_classes.png")
print("  • classification_report.txt")

Device: cpu
✅ 90 classes Food-101 chargées


Using cache found in C:\Users\pc/.cache\torch\hub\pytorch_vision_v0.10.0


RuntimeError: Error(s) in loading state_dict for ResNet:
	Missing key(s) in state_dict: "fc.weight", "fc.bias". 
	Unexpected key(s) in state_dict: "fc.0.weight", "fc.0.bias", "fc.3.weight", "fc.3.bias". 